## Data Sampling with a sliding window

In [1]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

tokenizer = tiktoken.get_encoding("gpt2")

In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

encoded_text = tokenizer.encode(raw_text)
print(len(encoded_text))

5145


In [3]:
enc_sample = encoded_text[50:]
print(len(enc_sample))

5095


In [4]:
CONTEXT_SIZE = 4
x = enc_sample[:CONTEXT_SIZE]
y = enc_sample[1:CONTEXT_SIZE+1]

print(f"x: {tokenizer.decode(x)}")
print(f"y: {tokenizer.decode(y)}")

x:  and established himself in
y:  established himself in a


In [5]:
for i in range(1, CONTEXT_SIZE+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "----->", desired)


print("\n\n ##### In English ##### \n\n")
for i in range(1, CONTEXT_SIZE+1):
    context = tokenizer.decode(enc_sample[:i])
    desired = tokenizer.decode([enc_sample[i]])

    print(context, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


 ##### In English ##### 


 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


In [6]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1: i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
        

In [7]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )

    return dataloader


In [8]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [12]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)


Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [11]:
inputs.shape

torch.Size([8, 4])

In [12]:
vocab_size = 50257
output_dim = 256
max_length = 4

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [13]:
token_embedding_layer.weight.shape 

torch.Size([50257, 256])

In [14]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [15]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [21]:
token_embeddings

tensor([[[ 0.2145,  0.8824,  0.3870,  ..., -0.8423,  1.1501,  0.4284],
         [-1.4276, -0.9401,  2.1740,  ...,  0.2812, -1.1846, -0.0888],
         [-0.6502,  0.7113,  0.5312,  ...,  0.9568, -1.8993, -1.0931],
         [ 0.9676,  0.7773,  0.1445,  ...,  0.2441, -0.3738,  1.2995]],

        [[-0.5092, -0.2736,  0.0666,  ..., -1.3695, -1.9833, -0.8516],
         [ 1.5900, -1.5000, -1.5077,  ..., -0.9877,  0.0301, -0.2924],
         [-1.1716, -0.1309,  1.5551,  ..., -0.4674,  0.0883,  0.1476],
         [-1.2710,  0.7236,  1.0914,  ..., -0.3922, -0.1035,  0.1289]],

        [[-0.1178, -0.0241, -0.4929,  ...,  2.1602, -0.0433,  0.5399],
         [-0.7197,  0.8559,  0.5031,  ..., -1.0086,  0.3875,  1.0892],
         [-0.8227,  0.9330,  0.0748,  ...,  1.0873, -0.0194,  0.5111],
         [-0.1148,  0.6566, -0.4504,  ...,  0.4197, -0.9352,  1.2763]],

        ...,

        [[ 0.5553,  1.8795, -0.1250,  ..., -0.4427,  0.5408, -0.4417],
         [-0.6965, -0.0992, -0.6836,  ..., -2.5452, -1.62

In [17]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [18]:
print(pos_embedding_layer.weight)
print(pos_embedding_layer.weight.shape)

Parameter containing:
tensor([[ 5.0019e-01, -7.0406e-01,  5.9161e-01,  ...,  5.6012e-02,
         -8.9465e-01,  1.1374e+00],
        [ 6.6767e-01,  1.0982e+00, -8.4528e-04,  ..., -1.4033e-01,
          1.2867e+00,  2.9139e-01],
        [ 3.0175e-01,  8.7461e-01,  2.6599e-01,  ...,  1.6307e+00,
         -5.7403e-02,  6.4257e-04],
        [-1.2065e+00, -1.2283e+00,  5.9821e-01,  ...,  5.4875e-02,
          3.2521e-01,  2.1290e+00]], requires_grad=True)
torch.Size([4, 256])


In [19]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [20]:
pos_embeddings

tensor([[ 5.0019e-01, -7.0406e-01,  5.9161e-01,  ...,  5.6012e-02,
         -8.9465e-01,  1.1374e+00],
        [ 6.6767e-01,  1.0982e+00, -8.4528e-04,  ..., -1.4033e-01,
          1.2867e+00,  2.9139e-01],
        [ 3.0175e-01,  8.7461e-01,  2.6599e-01,  ...,  1.6307e+00,
         -5.7403e-02,  6.4257e-04],
        [-1.2065e+00, -1.2283e+00,  5.9821e-01,  ...,  5.4875e-02,
          3.2521e-01,  2.1290e+00]], grad_fn=<EmbeddingBackward0>)